In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import pairwise_distances
from sklearn.utils.validation import check_is_fitted

from sklearn_extra.cluster import KMedoids

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC18__split__v1__kmedoids_per_class__v1"
CARPETA_DATASET = "CIC18__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG KMEDOIDS POR CLASE =====
MAX_SAMPLES_PER_CLASS = 10000
K_MEDOIDS_PER_CLASS = 10
METRIC = "euclidean"

# Para calcular pseudo-probabilidades desde distancias
DISTANCE_TO_PROBA_EPS = 1e-9

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/CIC18__split__v1__kmedoids_per_class__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(1341149, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,2,0,44751,3,13,6733,6000,1100,0,41677,...,250,3,0,0,0,0,0,0,0,1
1,37274,4,753825,754,1064,6266,18066,1424,184,20085,...,4725,278,72650,56255,70259,43755,32542,11323,36885,3
2,2,0,4380198,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,7
3,624,0,9183,1,3,3,1,3,0,3,...,3,1,0,0,0,0,0,0,0,0
4,2,0,66089,3,13,379,1459,225,0,488,...,156,3,0,0,0,0,0,0,0,2


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
1,360000
0,360000
2,159089
3,116159
4,115628
5,111820
6,75238
7,33125
8,7926


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (1341149, 54)
Shape y_train: (1341149,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmedoids_per_class", KMedoidsPerClassClassifier(
        k_medoids_per_class=K_MEDOIDS_PER_CLASS,
        metric=METRIC,
        random_state=RANDOM_STATE,
        distance_to_proba_eps=DISTANCE_TO_PROBA_EPS,
        max_samples_per_class=MAX_SAMPLES_PER_CLASS
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('kmedoids_per_class', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,k_medoids_per_class,10
,metric,'euclidean'
,random_state,42
,distance_to_proba_eps,1e-09


In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",

    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",

    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",

    "mcc": make_scorer(matthews_corrcoef),

    "roc_auc": "roc_auc_ovr_weighted",
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1
)

cv_results.keys()

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
             ^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/utils/_response.py", line 235, in _get_response_values
    raise ValueError(
ValueError: Pipeline should ei

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc', 'test_roc_auc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "roc_auc": cv_results["test_roc_auc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.859192,0.923762,0.859192,0.877177,0.564625,0.857718,0.579240,0.838125,NaN,14.472836,0.323029
1,2,0.834511,0.908924,0.834511,0.852190,0.553906,0.861886,0.570490,0.811965,NaN,15.004179,0.268331
2,3,0.862428,0.927554,0.862428,0.878422,0.567906,0.879744,0.582348,0.842912,NaN,14.048725,0.276640
3,4,0.805715,0.899294,0.805715,0.829377,0.547007,0.867610,0.562357,0.780914,NaN,16.978378,0.277552
4,5,0.854721,0.923412,0.854721,0.873003,0.569518,0.884586,0.583218,0.834040,NaN,14.380492,0.260992


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "scaler": "StandardScaler",
        "modelo": "KMedoidsPerClassClassifier",
        "k_medoids_per_class": K_MEDOIDS_PER_CLASS,
        "metric": METRIC
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC18__split__v1__kmedoids_per_class__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__train.csv',
 'shape_train': {'rows': 1341149, 'cols': 55},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'scaler': 'StandardScaler',
  'modelo': 'KMedoidsPerClassClassifier',
  'k_medoids_per_class': 10,
  'metric': 'euclidean'},
 'metricas_media': {'accuracy': 0.8433134658473342,
  'precision_weighted': 0.9165892403016912,
  'recall_weighted': 0.8433134658473342,
  'f1_weighted': 0.8620340116300318,
  'precision_macro': 0.5605923941643753,
  'recall_macro': 0.8703087263042502,
  'f1_macro': 0.5755306618141602,
  'mcc': 0.821591470945607,
  'roc_auc': nan,
  'fit_time': 14.976921892166137,
  'score_time': 0.2813088417053223},
 'metricas_std': {'accuracy': 0.023657522695045657,
  'precision_weighted': 0.012001130996921069,
  'recall_weighted': 0.023657522695045657,
  '

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print(f"ROC AUC             : {summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.843313 ± 0.023658

Precision weighted  : 0.916589 ± 0.012001
Recall weighted     : 0.843313 ± 0.023658
F1 weighted         : 0.862034 ± 0.021104

Precision macro     : 0.560592 ± 0.009730
Recall macro        : 0.870309 ± 0.011507
F1 macro            : 0.575531 ± 0.008920

MCC                 : 0.821591 ± 0.025643
ROC AUC             : nan ± nan

Fit time medio      : 14.976922
Score time medio    : 0.281309


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/CIC18__split__v1__kmedoids_per_class__v1/CIC18__split__v1__kmedoids_per_class__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/CIC18__split__v1__kmedoids_per_class__v1/CIC18__split__v1__kmedoids_per_class__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.859192,0.923762,0.859192,0.877177,0.564625,0.857718,0.579240,0.838125,NaN,14.472836,0.323029
1,2,0.834511,0.908924,0.834511,0.852190,0.553906,0.861886,0.570490,0.811965,NaN,15.004179,0.268331
2,3,0.862428,0.927554,0.862428,0.878422,0.567906,0.879744,0.582348,0.842912,NaN,14.048725,0.276640
3,4,0.805715,0.899294,0.805715,0.829377,0.547007,0.867610,0.562357,0.780914,NaN,16.978378,0.277552
4,5,0.854721,0.923412,0.854721,0.873003,0.569518,0.884586,0.583218,0.834040,NaN,14.380492,0.260992


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(335288, 55)


,DST_PORT,PROTOCOL,FLOW_DURATION,TOT_FWD_PKTS,TOT_BWD_PKTS,TOTLEN_FWD_PKTS,TOTLEN_BWD_PKTS,FWD_PKT_LEN_MAX,FWD_PKT_LEN_MIN,FWD_PKT_LEN_MEAN,...,INIT_BWD_WIN_BYTS,FWD_ACT_DATA_PKTS,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_MEAN,IDLE_MAX,IDLE_MIN,LABEL
0,4790,0,1343807,1,3,163,1,6,10,180,...,3,3,0,0,0,0,0,0,0,0
1,2,0,13905,3,13,371,1459,261,0,443,...,156,3,0,0,0,0,0,0,0,2
2,2,0,5725,3,13,2844,1459,137,0,4649,...,156,3,0,0,0,0,0,0,0,2
3,2,0,217626,3,13,230,1459,203,0,20004,...,156,3,0,0,0,0,0,0,0,2
4,78205,5,456956,756,1066,16372,72605,1778,443,134586,...,4725,278,72650,56255,70259,43755,32542,11323,36885,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,90000
1,90000
2,39772
3,29040
4,28907
5,27955
6,18810
7,8281
8,1982


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (335288, 54)
Shape y_test: (335288,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
inicio = time.time()

pipeline.fit(X_train, y_train)

fin = time.time()

print("Modelo final entrenado con todo el dataset train.")
print(f"Tiempo de entrenamiento: {fin - inicio:.2f} segundos")

Modelo final entrenado con todo el dataset train.
Tiempo de entrenamiento: 16.88 segundos


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted",
    labels=pipeline.named_steps["kmedoids_per_class"].classes_
)

Predicciones en test generadas.
Número de predicciones: 335288


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.8170140297296652,
 'precision_weighted': 0.8964087502687378,
 'recall_weighted': 0.8170140297296652,
 'f1_weighted': 0.8428948712579694,
 'precision_macro': 0.554483112291344,
 'recall_macro': 0.8572188153547348,
 'f1_macro': 0.563773408105526,
 'roc_auc': 0.961941503717925,
 'mcc': 0.7901041934371362}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")
print(f"ROC AUC             : {metricas_test['roc_auc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.817014

Precision weighted  : 0.896409
Recall weighted     : 0.817014
F1 weighted         : 0.842895

Precision macro     : 0.554483
Recall macro        : 0.857219
F1 macro            : 0.563773

MCC                 : 0.790104
ROC AUC             : 0.961942


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,47649,1014,3986,1853,1166,4335,371,5560,1313,1,4092,5104,12091,28,1437
1,5218,84088,495,0,0,0,0,0,0,0,193,5,1,0,0
2,139,2961,35795,0,0,0,1,839,0,0,36,0,1,0,0
3,67,0,0,26175,0,0,0,0,0,0,0,0,0,2798,0
4,0,0,0,0,28603,0,0,0,0,0,0,101,158,0,45
5,1325,0,0,2002,0,22929,0,171,1485,0,0,0,0,43,0
6,0,0,61,0,0,0,18495,128,0,0,57,64,0,0,5
7,37,13,10,0,0,0,0,7860,282,0,79,0,0,0,0
8,8,0,0,0,0,0,0,30,1855,0,53,8,27,0,1
9,0,0,0,0,0,0,0,0,0,342,4,0,0,0,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       0.88      0.53      0.66     90000
           1       0.95      0.93      0.94     90000
           2       0.89      0.90      0.89     39772
           3       0.87      0.90      0.89     29040
           4       0.96      0.99      0.97     28907
           5       0.84      0.82      0.83     27955
           6       0.98      0.98      0.98     18810
           7       0.54      0.95      0.69      8281
           8       0.38      0.94      0.54      1982
           9       1.00      0.99      0.99       346
          10       0.02      0.74      0.03       111
          11       0.01      0.72      0.01        46
          12       0.00      0.47      0.00        17
          13       0.00      1.00      0.01        11
          14       0.01      1.00      0.01        10

    accuracy                           0.82    335288
   macro avg       0.55      0.

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "scaler": "StandardScaler",
        "modelo": "KMedoidsPerClassClassifier",
        "k_medoids_per_class": K_MEDOIDS_PER_CLASS,
        "metric": METRIC
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "roc_auc": float(metricas_test["roc_auc"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'CIC18__split__v1__kmedoids_per_class__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_MEDOIDES/02_datasets/processed/CIC18__split__v1/CIC18__split__v1__test.csv',
 'shape_test': {'rows': 335288, 'cols': 55},
 'parametros': {'label_col': 'LABEL',
  'scaler': 'StandardScaler',
  'modelo': 'KMedoidsPerClassClassifier',
  'k_medoids_per_class': 10,
  'metric': 'euclidean'},
 'metricas_test': {'accuracy': 0.8170140297296652,
  'precision_weighted': 0.8964087502687378,
  'recall_weighted': 0.8170140297296652,
  'f1_weighted': 0.8428948712579694,
  'precision_macro': 0.554483112291344,
  'recall_macro': 0.8572188153547348,
  'f1_macro': 0.563773408105526,
  'roc_auc': 0.961941503717925,
  'mcc': 0.7901041934371362}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/CIC18__split__v1__kmedoids_per_class__v1/CIC18__split__v1__kmedoids_per_class__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/CIC18__split__v1__kmedoids_per_class__v1/CIC18__split__v1__kmedoids_per_class__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_MEDOIDES/04_experimentos/logs/resultados/CIC18__split__v1__kmedoids_per_class__v1/CIC18__split__v1__kmedoids_per_class__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.8433134658473342, 'precision_weighted': 0.9165892403016912, 'recall_weighted': 0.8433134658473342, 'f1_weighted': 0.8620340116300318, 'precision_macro': 0.5605923941643753, 'recall_macro': 0.8703087263042502, 'f1_macro': 0.5755306618141602, 'mcc': 0.821591470945607, 'roc_auc': nan, 'fit_time': 14.976921892166137, 'score_time': 0.2813088417053223}

TEST:
{'accuracy': 0.8170140297296652, 'precision_weighted': 0.8964087502687378, 'recall_weighted': 0.8170140297296652, 'f1_weighted': 0.8428948712579694, 'precision_macro': 0.554483112291344, 'recall_macro': 0.8572188153547348, 'f1_macro': 0.563773408105526, 'roc_auc': 0.961941503717925, 'mcc': 0.7901041934371362}
